In [ ]:
import altair as alt
import pandas as pd
import geopandas as gpd # Requires geopandas -- e.g.: conda install -c conda-forge geopandas
alt.data_transformers.enable('json') # Let Altair/Vega-Lite work with large data sets

pass

In [8]:
names = pd.read_csv("dpt2020.csv", sep=";")
names.drop(names[names.preusuel == '_PRENOMS_RARES'].index, inplace=True)
names.drop(names[names.dpt == 'XX'].index, inplace=True)

names.sample(5)

,sexe,preusuel,annais,dpt,nombre
2175199,2,CLARA,2020,68,5
2249659,2,DANIELLE,1962,81,13
793394,1,JEAN-CLAUDE,1952,21,65
1056978,1,LUDOVIC,1971,66,13
3126235,2,MARIETTE,1913,01,5


In [9]:
depts = gpd.read_file('departements-version-simplifiee.geojson')

depts.sample(5)

,code,nom,geometry
61,61,Orne,"POLYGON ((-0.84094 48.75222, -0.81927 48.75413..."
93,93,Seine-Saint-Denis,"POLYGON ((2.55306 49.00982, 2.58031 48.99159, ..."
70,70,Haute-Saône,"POLYGON ((5.88473 47.92605, 5.90011 47.94475, ..."
53,53,Mayenne,"POLYGON ((-1.07016 48.50849, -1.06055 48.51534..."
72,72,Sarthe,"POLYGON ((-0.05453 48.382, -0.04463 48.37976, ..."


In [10]:
# Keep a reference around to the plain pandas dataframe, without geometry data, just in case
just_names = names

names = depts.merge(names, how='right', left_on='code', right_on='dpt')

names.sample(5)

,code,nom,geometry,sexe,preusuel,annais,dpt,nombre
3494336,22,Côtes-d'Armor,"POLYGON ((-3.65914 48.65921, -3.63649 48.67069...",2,SIMONE,1915,22,50
1524018,56,Morbihan,"MULTIPOLYGON (((-3.42179 47.62, -3.44067 47.62...",1,SWANN,1998,56,5
1875983,58,Nièvre,"POLYGON ((2.87463 47.52042, 2.8489 47.53754, 2...",2,ANNE-LAURE,1984,58,6
1896090,NaN,NaN,None,2,ANNIE,1972,974,38
308447,38,Isère,"POLYGON ((5.62375 45.61327, 5.62303 45.60428, ...",1,CLÉMENT,1948,38,3


In [11]:
#objectif 1 : chart des 20 prénoms les plus populaires de 2004
"""
annais : filtrer sur 2004
puis garder que les 20 premiers de nombre
puis faire le chart
"""

'\nannais : filtrer sur 2004\npuis garder que les 20 premiers de nombre\npuis faire le chart\n'

In [12]:
names_2004 = just_names[just_names["annais"]=="2004"]

names_2004.head(10)

,sexe,preusuel,annais,dpt,nombre
10987,1,AARON,2004,02,4
10988,1,AARON,2004,03,3
10989,1,AARON,2004,08,4
10990,1,AARON,2004,12,3
10991,1,AARON,2004,13,11
10992,1,AARON,2004,14,4
10993,1,AARON,2004,21,3
10994,1,AARON,2004,25,4
10995,1,AARON,2004,29,4
10996,1,AARON,2004,31,8


In [13]:
grouped_2004 = names_2004.groupby(['preusuel', 'sexe'], as_index=False).sum(numeric_only=True)

grouped_2004


,preusuel,sexe,nombre
0,AALIYAH,2,29
1,AARON,1,281
2,ABD,1,3
3,ABDALLAH,1,82
4,ABDEL,1,22
...,...,...,...
3633,ZOHRA,2,18
3634,ZORAN,1,3
3635,ZOUMANA,1,3
3636,ZOÉ,2,2280


In [14]:
ordered_2004 = grouped_2004.sort_values(by=["nombre"], ascending=False)

In [15]:
top_20 = ordered_2004.iloc[0:20]

In [16]:
base = alt.Chart(top_20).mark_bar().encode(
    x = 'nombre:Q',
    y =alt.Y('preusuel:N', sort='-x')
)

base

alt.Chart(...)

In [17]:
slider = alt.binding_range(min=1900, max=2020, step=1, name='year:')
year = alt.param(value=2004, bind=slider)
base.add_params(year)

alt.Chart(...)

In [18]:
def table(year): 
    names_2004 = just_names[just_names["annais"]==str(year)]
    grouped_2004 = names_2004.groupby(['preusuel', 'sexe'], as_index=False).sum(numeric_only=True)
    ordered_2004 = grouped_2004.sort_values(by=["nombre"], ascending=False)
    top_20 = ordered_2004.iloc[0:20]
    return top_20


In [19]:
just_names

,sexe,preusuel,annais,dpt,nombre
10885,1,AADIL,1983,84,3
10886,1,AADIL,1992,92,3
10888,1,AAHIL,2016,95,3
10892,1,AARON,1962,75,3
10893,1,AARON,1976,75,3
...,...,...,...,...,...
3727545,2,ZYA,2013,44,4
3727546,2,ZYA,2013,59,3
3727547,2,ZYA,2017,974,3
3727548,2,ZYA,2018,59,3


In [20]:
just_names_year = just_names.groupby(['annais', 'preusuel', 'sexe'], as_index=False).sum(numeric_only=True)
# just_names_ordered = just_names_year.sort_values(by=["nombre"], ascending=False)


just_names_year.head(20)



,annais,preusuel,sexe,nombre
0,1900,ABEL,1,382
1,1900,ABRAHAM,1,9
2,1900,ACHILLE,1,152
3,1900,ACHILLES,1,4
4,1900,ADAM,1,9
5,1900,ADELAIDE,2,143
6,1900,ADELHEID,2,3
7,1900,ADELINA,2,27
8,1900,ADELINE,2,169
9,1900,ADOLPHE,1,464


In [21]:
# slider = alt.binding_range(min=1900, max=2020, step=1, name='year:')
# year = alt.param(value=2004, bind=slider)



# selection = alt.selection_point(fields=["annais"], bind=slider)
# color = (
#     alt.when(selection)
#     .then(alt.Color("Origin:N").legend(None))
#     .otherwise(alt.value("lightgray"))
# )

# base = alt.Chart(just_names.iloc[0:100]).mark_bar().encode(
#     x = 'nombre:Q',
#     y =alt.Y('preusuel:N', sort='-x'),
#     color=color
# ).add_params(selection)

# base




# base.add_params(year)
